Install necessary libraries, if need

In [1]:
# pip install catboost polars pandas

Importing of necessary libraries

In [8]:
import polars as pl
import polars.selectors as cs
import pandas as pd
from catboost import CatBoostClassifier

These are crops with the corresponding numbers

In [38]:
class_names = {
    '1': 'winter wheat',
    '2': 'spring oats',
    '3': 'spring barley',
    '4': 'spring rye',
    '5': 'corn',
    '6': 'soybean',
    '7': 'sunflower',
    '8': 'sugar beet',
    '9': 'rapeseed',
    '10': 'sorghum',
    '11': 'potato',
    '12': 'cotton',
    '13': 'spring wheat',
    '14': 'winter oats',
    '15': 'winter barley',
    '16': 'winter rye'
}

Load input dataset with meteorological, spectral and phenological data. The dataset consists of columns: 'field_id' - number of field and all the predictors.

In [39]:
df = pl.read_parquet('../data/processed/2015.parquet.gzip')
df

field_id,wrdvi_wNDVI,wrdvi_mNDVI,wrdvi_S,wrdvi_A,wrdvi_mS,wrdvi_mA,wrdvi_doy_max,wrdvi_max,wrdvi_start_of_growth,wrdvi_end_of_growth,wrdvi_start_of_decay,wrdvi_end_of_decay,wrdvi_max_growth,wrdvi_mean_growth,wrdvi_min_growth,wrdvi_max_decay,wrdvi_min_decay,wrdvi_mean_decay,median_wrdvi_fitted_4,median_wrdvi_fitted_5,median_wrdvi_fitted_6,median_wrdvi_fitted_7,median_wrdvi_fitted_8,median_wrdvi_fitted_9,median_wrdvi_fitted_10,ndre_wNDVI,ndre_mNDVI,ndre_S,ndre_A,ndre_mS,ndre_mA,ndre_doy_max,ndre_max,ndre_start_of_growth,ndre_end_of_growth,ndre_start_of_decay,…,swir2_doy_max,swir2_max,median_swir2_fitted_4,median_swir2_fitted_5,median_swir2_fitted_6,median_swir2_fitted_7,median_swir2_fitted_8,median_swir2_fitted_9,median_swir2_fitted_10,median_t_9,sum_prec_9,sum_prec_6,median_prec_7,sum_t_4,sum_t_7,sum_prec_5,sum_t_5,median_t_8,median_t_5,sum_t_8,median_prec_4,median_t_7,median_t_10,median_t_4,median_prec_8,sum_prec_4,sum_t_9,median_prec_10,median_prec_6,sum_prec_7,median_prec_9,sum_prec_8,sum_prec_10,median_prec_5,sum_t_6,sum_t_10,median_t_6
i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
6,-0.770454,-0.212571,115.470013,144.104936,0.372099,0.070134,124.63982,-0.343853,85.672336,121.544272,176.717859,176.717859,-0.360379,-0.55753,-0.779557,-0.719027,-0.719027,-0.491526,-0.78465,-0.409904,-0.671464,-0.756633,-0.768852,-0.770265,-0.770432,-0.071034,0.541566,114.02855,157.747553,0.510073,0.066308,122.818909,0.479714,88.585793,118.448724,135.01901,…,224.971986,1.593512,1.147506,1.038956,1.068016,1.374877,1.582877,1.504957,1.478267,294.900911,0.030391,0.062568,0.000004,8481.958671,9239.902961,0.187671,8927.05987,298.828662,287.717198,9267.680447,0.000183,297.593664,284.353228,282.381261,0.000008,0.046016,8797.806135,0.000139,0.000498,0.019522,6.4096e-7,0.03852,0.085601,0.003177,8780.852101,8803.422937,292.584293
12,-0.781273,-0.578835,117.140097,175.106576,0.50615,0.058058,127.189095,-0.591881,89.86043,121.544272,141.392196,214.410705,-0.60715,-0.686814,-0.782698,-0.603887,-0.762521,-0.680054,-0.783011,-0.599913,-0.653832,-0.735859,-0.772059,-0.779643,-0.780994,-0.095102,0.202083,132.616109,192.226889,0.999929,0.141245,141.756378,0.201813,113.532266,134.836918,175.989495,…,1.0,2.978719,1.683658,1.303759,1.119775,1.341017,1.52643,1.52124,1.584179,294.718741,0.030603,0.067389,0.00001,8477.593192,9228.564582,0.182001,8920.186579,298.755013,287.506403,9261.023058,0.000245,297.361958,284.196395,282.19254,0.000006,0.048273,8791.273908,0.000124,0.000895,0.020659,6.4096e-7,0.042608,0.087821,0.0025,8772.371632,8798.792973,292.252506
9,-0.75942,-0.539249,110.664507,163.327541,0.095795,0.465224,151.407204,-0.544461,86.582791,134.290645,158.326663,194.016508,-0.559992,-0.649334,-0.739483,-0.561099,-0.759495,-0.650744,-0.678618,-0.557958,-0.710771,-0.759479,-0.759423,-0.75942,-0.75942,-0.060765,1.0,125.301411,153.98077,0.06464,0.062579,139.935468,0.392075,86.218609,123.547274,155.595298,…,208.947974,1.605587,1.040488,0.968074,1.040643,1.442066,1.525737,1.499488,1.387988,295.043245,0.029706,0.056481,0.000003,8486.13022,9252.424283,0.173089,8933.039687,299.032357,288.051915,9275.888211,0.000124,297.887823,284.397825,282.579321,0.000005,0.044621,8805.562808,0.000136,0.000404,0.019922,6.4096e-7,0.032824,0.081442,0.002767,8790.043263,8808.059656,292.849102
0,-0.783048,-0.539082,123.84396,173.288268,0.0562,0.082034,149.222111,-0.616081,82.030515,129.738369,173.076038,339.325163,-0.647707,-0.665217,-0.761942,-0.674431,-0.783049,-0.675332,-0.721191,-0.633057,-0.64643,-0.755412,-0.780946,-0.782944,-0.783052,-0.080181,0.194169,12.915214,186.250734,0.995,0.2285,46.158579,0.194169,10.468734,15.203102,176.171586,…,365.0,1.977999,1.100109,0.976382,1.08049,1.407656,1.561645,1.450324,1.545586,294.528353,0.030959,0.06

# CropGRM-large

Selecting predictors for CropGRM-large

In [ ]:
suffixes = [
    '_wNDVI', '_mNDVI', '_S', '_A', '_mS', '_mA',
    '_doy_max', '_doy_min', '_start_of_growth', '_end_of_growth',
    '_start_of_decay', '_end_of_decay',
    '_max_growth', '_mean_growth', '_min_growth',
    '_max_decay', '_min_decay', '_mean_decay'
]

indexes=['red', 'nir', 'swir1', 'swir2', 'green', 'blue', 'wrdvi', 'ndre', 'ndyi', 'median_red', 'median_nir', 'median_swir1', 'median_swir2', 'median_blue', 'median_green']
matched_cols = (df.select(cs.ends_with(suffixes)))
matched_cols = (df.select(cs.starts_with(indexes)))
cols_to_select = [f"sum_t_{i}" for i in range(4, 11)] + [f"sum_prec_{i}" for i in range(4, 11)] + [f"median_t_{i}" for i in range(4, 11)] + [f"median_prec_{i}" for i in range(4, 11)]
cols_to_select += [*matched_cols.columns]

sample = df.drop_nulls(subset=cols_to_select)
temp_df = sample.to_pandas()
pred_features=temp_df[cols_to_select]

Loading of model CropGRM-large

In [40]:
model = CatBoostClassifier()
model.load_model('../models/CropGRM-large.cbm')

Saving the prediction results into csv

In [ ]:
y_pred = model.predict(pred_features)
temp_df['class']=y_pred
temp_df['class'] = temp_df['class'].astype(str)
temp_df['class_name']=temp_df['class'].map(class_names)
temp_df[['field_id', 'class_name']].to_csv('../data/final/2015_CropGRM-large_predictions.csv', index=False)

# CropGRM-optimized

Selecting predictors for CropGRM-optimized

In [41]:
cols_to_select=["sum_t_4", "sum_t_5", "sum_t_6", "sum_t_7", "sum_t_8", "sum_t_9", "sum_t_10",
    "sum_prec_4", "sum_prec_5", "sum_prec_6", "sum_prec_7", "sum_prec_8", "sum_prec_9", "sum_prec_10",
    "median_t_4", "median_t_5", "median_t_6", "median_t_7", "median_t_8", "median_t_9", "median_t_10",
    "median_prec_4", "median_prec_5", "median_prec_6", "median_prec_7", "median_prec_8", "median_prec_9", "median_prec_10",
    "wrdvi_wNDVI", "wrdvi_S", "wrdvi_A", "wrdvi_mS", "wrdvi_mA", "wrdvi_max", "wrdvi_end_of_growth",
    "ndre_wNDVI", "ndre_S", "ndre_A", "ndre_mS", "ndre_mA", "ndre_max", "ndyi_doy_max", "ndyi_max",
    "red_min", "red_doy_min", "median_red_fitted_4", "median_red_fitted_5", "median_red_fitted_6", 
    "median_red_fitted_7", "median_red_fitted_8", "median_red_fitted_9", "median_red_fitted_10",
    "nir_max", "median_nir_fitted_4", "median_nir_fitted_5", "median_nir_fitted_6", "median_nir_fitted_7",
    "median_nir_fitted_8", "median_nir_fitted_9", "median_nir_fitted_10", "median_blue_fitted_5",
    "median_blue_fitted_7", "median_blue_fitted_8", "median_blue_fitted_9", "median_swir1_fitted_5",
    "median_swir1_fitted_6", "median_swir1_fitted_7", "median_swir1_fitted_8", "median_swir1_fitted_9",
    "median_green_fitted_5", "median_green_fitted_6", "median_green_fitted_7", "median_green_fitted_8",
    "median_green_fitted_9", "swir2_min", "median_swir2_fitted_4", "median_swir2_fitted_5", "median_swir2_fitted_6",
    "median_swir2_fitted_7", "median_swir2_fitted_8", "median_swir2_fitted_9", "median_swir2_fitted_10"
]

sample = df.drop_nulls(subset=cols_to_select)
temp_df = sample.to_pandas()
pred_features=temp_df[cols_to_select]

Loading of model CropGRM-optimized

In [ ]:
model = CatBoostClassifier()
model.load_model('../models/CropGRM-optimized.cbm')

Saving the prediction results into csv

In [ ]:
y_pred = model.predict(pred_features)
temp_df['class']=y_pred
temp_df['class'] = temp_df['class'].astype(str)
temp_df['class_name']=temp_df['class'].map(class_names)
temp_df[['field_id', 'class_name']].to_csv('../data/final/2015_CropGRM-optimized_predictions.csv', index=False)

# CropGRM-small

Selecting predictors for CropGRM-small

In [42]:
cols_to_select=  ['sum_t_4', 'sum_t_5', 'sum_t_6', 'sum_t_7', 'sum_t_8', 'sum_t_9', 'sum_t_10', 
                  'sum_prec_4', 'sum_prec_6', 'sum_prec_10', 'median_t_4', 'median_t_6', 'median_t_9', 
                  'median_t_10', 'ndre_S', 'median_red_fitted_8', 'median_nir_fitted_5', 'median_nir_fitted_8', 
                  'median_swir1_fitted_6', 'median_swir1_fitted_7', 'median_swir1_fitted_8', 'median_green_fitted_7', 
                  'median_green_fitted_8', 'median_swir2_fitted_5']

sample = df.drop_nulls(subset=cols_to_select)
temp_df = sample.to_pandas()
pred_features=temp_df[cols_to_select]

Loading of model CropGRM-small

In [ ]:
model = CatBoostClassifier()
model.load_model('../models/CropGRM-small.cbm')

Saving the prediction results into csv

In [ ]:
y_pred = model.predict(pred_features)
temp_df['class']=y_pred
temp_df['class'] = temp_df['class'].astype(str)
temp_df['class_name']=temp_df['class'].map(class_names)
temp_df[['field_id', 'class_name']].to_csv('../data/final/2015_CropGRM-small_predictions.csv', index=False)